In [23]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

In [24]:
num_samples = 1000
num_features = 10
num_classes = 3

In [25]:
# Generate random input data (features)
x_train = np.random.random((num_samples, num_features)).astype(np.float32)
# Generate random target data (labels)
y_train = np.random.randint(0, num_classes, size=(num_samples,))
y_train = tf.keras.utils.to_categorical(y_train, num_classes)


In [26]:
# Step 2: Define and compile a simple model
model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(num_features,)),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [27]:
# Step 3: Train the model
model.fit(x_train, y_train, epochs=5, batch_size=32)


Epoch 1/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.3310 - loss: 1.1118   
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3412 - loss: 1.0999 
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3770 - loss: 1.0956 
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3646 - loss: 1.0909 
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3717 - loss: 1.0919 


In [28]:
# Step 4: Convert the trained model to TensorFlow Lite format with quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable post-training quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]

In [29]:
# Convert the model
quantized_model = converter.convert()

# Step 5: Save the quantized model
with open('quantized_model.tflite', 'wb') as f:
    f.write(quantized_model)

print("Model successfully quantized and saved as 'quantized_model.tflite'.")


Saved artifact at '/tmp/tmpawmuwlk8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 10), dtype=tf.float32, name='keras_tensor_18')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  134564407624272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134564406729664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134564406736528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134564406730368: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model successfully quantized and saved as 'quantized_model.tflite'.


In [30]:
# Step 6: Load and run the quantized model for inference (optional)
interpreter = tf.lite.Interpreter(model_path='quantized_model.tflite')
interpreter.allocate_tensors()

# Get input and output tensors
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [31]:
# Prepare a sample input for inference
sample_input = np.random.random((1, num_features)).astype(np.float32)

# Set the input tensor
interpreter.set_tensor(input_details[0]['index'], sample_input)

# Run inference
interpreter.invoke()

In [32]:
# Get the output tensor
output = interpreter.get_tensor(output_details[0]['index'])
print(f"Quantized model output: {output}")


Quantized model output: [[0.39884314 0.3381458  0.263011  ]]
